# Faz 1 — Veri Seti Seçimi ve Kısıt Doğrulaması

**Veri seti:** Diabetes 130-US Hospitals (1999–2008), UCI ML Repository

**Hedef:** Hastaneye yeniden yatış — Sınıf 1: `<30`, Sınıf 0: diğerleri

| Kriter | Şart |
|---|---|
| Satır | ≥ 20.000 |
| Özellik | ≥ 20 |
| Kategorik oran | ≥ %30 |
| Dengesizlik | ≥ 1:3 |

In [1]:
from src import config
from src.data_loader import add_binary_target, load_raw_data
from src.validate_constraints import (
    check_constraints,
    get_categorical_columns,
    get_feature_columns,
)

## 1. Veriyi pandas ile okuma

In [2]:
df = add_binary_target(load_raw_data())
print(f'Boyut: {df.shape}')
df.head()

Boyut: (101766, 51)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,readmitted_30d
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,NO,0
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,Up,No,No,No,No,No,Ch,Yes,>30,0
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,Yes,NO,0
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,Up,No,No,No,No,No,Ch,Yes,NO,0
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,Steady,No,No,No,No,No,Ch,Yes,NO,0


## 2. Kısıt doğrulama

In [3]:
report = check_constraints(df)
report

,Kriter,Değer,Şart,Sağlandı
0,Satır sayısı,101766,">= 20,000",True
1,Özellik sayısı,47,>= 20,True
2,Kategorik oran,83.0%,>= 30%,True
3,Dengesizlik (Sınıf 1 : Sınıf 0),1:7.96,>= 1:3,True


In [4]:
assert report['Sağlandı'].all(), 'Kısıtlardan biri sağlanmadı!'
print('Dört kısıtın tamamı sağlandı.')

Dört kısıtın tamamı sağlandı.


## 3. Detaylar

In [5]:
features = get_feature_columns(df)
categorical = get_categorical_columns(df, features)
numeric = [c for c in features if c not in categorical]
print(f'Toplam özellik: {len(features)}')
print(f'Kategorik ({len(categorical)}): {categorical}')
print(f'Sayısal   ({len(numeric)}): {numeric}')

Toplam özellik: 47
Kategorik (39): ['race', 'gender', 'age', 'weight', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id']
Sayısal   (8): ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']


In [6]:
df[config.TARGET].value_counts()

readmitted_30d
0    90409
1    11357
Name: count, dtype: int64